# AF2 recovered-cue class calibration — seed 42
Melatih hanya **189 bobot kalibrasi kelas** selama 20 epoch. AF2 asli, backbone, neck, classifier, dan box head dibekukan. Cue spasial yang sudah dihitung AF2 dipakai ulang; tidak ada FFT kedua, ROI, atau test.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import importlib, json, os, shutil, subprocess, sys, tarfile, time, torch
from pathlib import Path
assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
REPO=Path('/content/coffee-bean-detection'); BRANCH='codex/af2-recovered-cue-calibration'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96','-e',str(REPO)],check=True)
for module_name in list(sys.modules):
    if module_name=='coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name,None)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
from coffee_detector.drive_project import resolve_drive_project_root,require_project_artifact
REQ=(
 'bundles/faruq-development-v3-grouped.tar',
 'experiments/faruq-v3-breadth-screening-batch-v1/candidates/AFAB/AF2_seed42/weights/best.pt',
)
PROJECT=resolve_drive_project_root(required_relative_paths=REQ)
ARCHIVE=require_project_artifact(PROJECT,REQ[0]); AF2=require_project_artifact(PROJECT,REQ[1])
DATA=Path('/content/faruq-development-v3-grouped')
if not (DATA/'data.yaml').is_file():
    with tarfile.open(ARCHIVE,'r') as archive: archive.extractall('/content',filter='data')
assert (DATA/'train/images').is_dir() and (DATA/'val/images').is_dir()
assert not (DATA/'test').exists(), 'STOP: test tersedia.'
GROUPED=DATA/'faruq_grouped_summary.json'; assert GROUPED.is_file(),GROUPED
OUTPUT=PROJECT/'experiments/faruq-v3-af2-recovered-cue-calibration-v1'; OUTPUT.mkdir(parents=True,exist_ok=True)
STATIC=OUTPUT/'static_audit.json'
print('GPU:',torch.cuda.get_device_name(0)); print('PROJECT:',PROJECT); print('AF2:',AF2); print('OUTPUT:',OUTPUT)

In [ ]:
from coffee_detector.af2_rcc import run_af2_rcc_static_audit
audit=run_af2_rcc_static_audit(AF2,STATIC,device='cuda:0')
print('PARAMETERS:',{'source':audit['source_parameters'],'candidate':audit['candidate_parameters'],'added':audit['added_parameters']})
print('GATES:',audit['gates']); print('DECISION:',audit['decision'])
assert audit['decision']=='PASS','STOP: wiring/freeze AF2-RCC tidak aman; jangan training.'

In [ ]:
RESULT=OUTPUT/'val_reports/AF2RCC1_seed42_result.json'; LOG=OUTPUT/'AF2RCC1_seed42_run.log'
if RESULT.is_file():
    print('REUSE COMPLETE:',RESULT)
else:
    command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_rcc_arm','--data-root',str(DATA),'--grouped-summary',str(GROUPED),'--af2-checkpoint',str(AF2),'--static-audit',str(STATIC),'--output-root',str(OUTPUT),'--seed','42','--device','0','--authorize-training']
    print('START/RESUME AF2RCC1 | log=',LOG,flush=True)
    with LOG.open('a',encoding='utf-8') as stream:
        process=subprocess.Popen(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT,text=True)
    seen=None
    while process.poll() is None:
        csv=OUTPUT/'AF2RCC1/AF2RCC1_seed42/results.csv'
        epochs=max(0,len(csv.read_text(errors='replace').splitlines())-1) if csv.is_file() else 0
        if epochs!=seen: print(f'AF2RCC1: {epochs}/20 epoch tercatat',flush=True); seen=epochs
        time.sleep(60)
    if process.returncode:
        print('\n'.join(LOG.read_text(errors='replace').splitlines()[-180:])); raise RuntimeError(f'AF2RCC1 gagal: {process.returncode}')
assert RESULT.is_file(),RESULT
result=json.loads(RESULT.read_text())
print('AF2:',{key:result['baseline_metrics'][key] for key in ('macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')})
print('AF2RCC1:',{key:result['metrics'][key] for key in ('macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')})

In [ ]:
from coffee_detector.experiments.run_faruq_v3_af2_rcc_decision import run_faruq_v3_af2_rcc_decision
DECISION_PATH=OUTPUT/'val_reports/af2_rcc_seed42_decision.json'
decision=run_faruq_v3_af2_rcc_decision(RESULT,DECISION_PATH)
print('DECISION:',decision['decision']); print('NEXT:',decision['next'])
print('Kirim baseline, candidate, deltas, target_class_delta, criteria, dan decision. Jangan buka test atau seed lain jika FAIL.')